In [ ]:
# === Setup ===
# Runtime: <2m with OAI_FAST_MODE=1
# Hardware: CPU smoke; GPU recommended for full run
# Network: none
# Competition-safe: Yes for the declared profile
import os, random, math, re, json, csv, time
from pathlib import Path
import numpy as np
import torch
torch.manual_seed(42)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
FAST_MODE = os.getenv("OAI_FAST_MODE", "0") == "1"
RUNTIME_PROFILE = os.getenv("OAI_RUNTIME_PROFILE", "cpu")
DEVICE = torch.device("cuda" if RUNTIME_PROFILE == "gpu" else "cpu")
if RUNTIME_PROFILE == "gpu" and not torch.cuda.is_available(): raise RuntimeError("GPU profile requested but CUDA is unavailable")
torch.set_default_device(DEVICE)
_device_probe = (torch.ones(8, device=DEVICE) @ torch.ones(8, device=DEVICE)).item()
print(f"Compute device: {DEVICE}; probe={_device_probe:.1f}")
random.seed(42)
np.random.seed(42)
print(f"Runtime profile: {'fast' if FAST_MODE else 'full'}")

# Experiment — bỏ positional encoding

**Hypothesis:** không có position, self-attention thuần là permutation-equivariant.

In [ ]:
torch.manual_seed(42); m=torch.nn.MultiheadAttention(8,2,batch_first=True,dropout=0.); x=torch.randn(1,4,8); perm=torch.tensor([2,0,3,1])
y,_=m(x,x,x); yp,_=m(x[:,perm],x[:,perm],x[:,perm]); err=(yp-y[:,perm]).abs().max().item()
assert err<1e-5; print("permutation equivariance error",err)

**Observation:** attention không tự biết thứ tự; positional information phá đối xứng hoán vị.